# Schedules: payment grids and observation grids

How a leg's dates are built, from the bare `generate_schedule` up to the two grids the pricer reads:

1. **Payment grid** (`build_payment_schedule`): accrual periods, business-day adjustment, payment delay, stubs, period fractions.
2. **Single-fixing reset windows**: which rate window a simple floating period reads (in arrears vs in advance, slower resets).
3. **Observation grid** (`build_observation_grid`): the daily sub-periods a compounded or averaged coupon reads, with lookback, observation shift and lockout.
4. Intra-period term fixings (monthly fixings inside a quarterly coupon).
5. How the kernel consumes the grid (flatten and `reduceat`).

Run from the repo root with `PYTHONPATH=src`.

In [1]:
import numpy as np
import pandas as pd

from finance.dates import BDC, Date, DayCountMethod, Frequency, Term
from finance.dates.calendars import Calendars
from finance.dates.calendars.calendar import StaticCalendar
from finance.dates.enums import FixingType
from finance.dates.term.term_type import TermType
from finance.instruments.schedules import build_payment_schedule
from finance.instruments.schedules.observation import LookbackStyle, build_observation_grid

pd.set_option("display.width", 160)

# A demo calendar: Monday-Friday plus a few holidays that land inside the schedules below,
# so business-day adjustment has something to do.  Only "no_holidays" ships with the library.
Calendars.register(StaticCalendar(
    name="demo",
    week_mask=[1, 1, 1, 1, 1, 0, 0],
    holidays={Date(2026, 9, 7), Date(2027, 6, 1), Date(2028, 12, 1)},   # Labor Day '26, two made-up roll-date holidays
))


def frame(ps, *columns) -> pd.DataFrame:
    """A PaymentSchedule's per-period arrays as a table."""
    data = {
        "accrual_start": ps.accrual_starts,
        "accrual_end": ps.accrual_ends,
        "payment": ps.payment_dates,
        "days": (ps.accrual_ends - ps.accrual_starts).astype(int),
        "frac": ps.period_fracs.round(6),
    }
    if ps.reset_starts is not None:
        data["reset_start"] = ps.reset_starts
        data["reset_end"] = ps.reset_ends
    df = pd.DataFrame(data)
    for col in ("accrual_start", "accrual_end", "payment", "reset_start", "reset_end"):
        if col in df:
            df[col] = pd.to_datetime(df[col]).dt.strftime("%Y-%m-%d  %a")
    return df[list(columns)] if columns else df

## 1. The payment grid

An annual fixed leg, Act/360, Modified Following on the demo calendar. The roll dates come from `generate_schedule`; the accrual boundaries are then adjusted to good business days, and the payment date is the adjusted accrual end.

2027-06-01 is a holiday on the demo calendar, 2030-06-01 is a Saturday: both roll to the next business day, and the period fractions follow the adjusted dates.

In [2]:
annual = build_payment_schedule(
    Date(2026, 6, 1), Date(2031, 6, 1), Frequency.Annually,
    DayCountMethod.Actual360, BDC.ModifiedFollowing, "demo",
)
frame(annual)

,accrual_start,accrual_end,payment,days,frac,reset_start,reset_end
0,2026-06-01 Mon,2027-06-02 Wed,2027-06-02 Wed,366,1.016667,2026-06-01 Mon,2027-06-02 Wed
1,2027-06-02 Wed,2028-06-01 Thu,2028-06-01 Thu,365,1.013889,2027-06-02 Wed,2028-06-01 Thu
2,2028-06-01 Thu,2029-06-01 Fri,2029-06-01 Fri,365,1.013889,2028-06-01 Thu,2029-06-01 Fri
3,2029-06-01 Fri,2030-06-03 Mon,2030-06-03 Mon,367,1.019444,2029-06-01 Fri,2030-06-03 Mon
4,2030-06-03 Mon,2031-06-02 Mon,2031-06-02 Mon,364,1.011111,2030-06-03 Mon,2031-06-02 Mon


### Payment delay and the choice of convention

A payment delay shifts the payment date only; the accrual period is unchanged. `Following` versus `ModifiedFollowing` differ only when rolling forward would cross a month end (2030-06-01, a Saturday, under `Preceding` moves back to Friday the 31st of May).

In [3]:
delayed = build_payment_schedule(
    Date(2026, 6, 1), Date(2031, 6, 1), Frequency.Annually,
    DayCountMethod.Actual360, BDC.ModifiedFollowing, "demo",
    payment_delay=Term(2, TermType.BusinessDays),  # pay two business days after the accrual end
)
preceding = build_payment_schedule(
    Date(2026, 6, 1), Date(2031, 6, 1), Frequency.Annually,
    DayCountMethod.Actual360, BDC.Preceding, "demo",
)
pd.DataFrame({
    "accrual_end (MF)": frame(annual)["accrual_end"],
    "payment (MF)": frame(annual)["payment"],
    "payment (MF + 2B delay)": frame(delayed)["payment"],
    "accrual_end (Preceding)": frame(preceding)["accrual_end"],
})

,accrual_end (MF),payment (MF),payment (MF + 2B delay),accrual_end (Preceding)
0,2027-06-02 Wed,2027-06-02 Wed,2027-06-04 Fri,2027-05-31 Mon
1,2028-06-01 Thu,2028-06-01 Thu,2028-06-05 Mon,2028-06-01 Thu
2,2029-06-01 Fri,2029-06-01 Fri,2029-06-05 Tue,2029-06-01 Fri
3,2030-06-03 Mon,2030-06-03 Mon,2030-06-05 Wed,2030-05-31 Fri
4,2031-06-02 Mon,2031-06-02 Mon,2031-06-04 Wed,2031-05-30 Fri


### Stubs

`first_regular` and `last_regular` pin the regular roll cycle, leaving a short (or long) stub at the front or back. The stub period has its own fraction, so a 2-month first period on a quarterly leg accrues for 2 months.

In [4]:
front_stub = build_payment_schedule(
    Date(2026, 7, 15), Date(2028, 9, 1), Frequency.Quarterly,
    DayCountMethod.Actual360, BDC.ModifiedFollowing, "demo",
    first_regular=Date(2026, 9, 1),                 # short front stub: 15 Jul -> 1 Sep, then regular quarters
)
back_stub = build_payment_schedule(
    Date(2026, 6, 1), Date(2028, 7, 15), Frequency.Quarterly,
    DayCountMethod.Actual360, BDC.ModifiedFollowing, "demo",
    last_regular=Date(2028, 6, 1),                  # regular quarters to 1 Jun 2028, then a short back stub
)
print("front stub:"); display(frame(front_stub).head(3))
print("back stub:");  display(frame(back_stub).tail(3))

front stub:


,accrual_start,accrual_end,payment,days,frac,reset_start,reset_end
0,2026-07-15 Wed,2026-09-01 Tue,2026-09-01 Tue,48,0.133333,2026-07-15 Wed,2026-09-01 Tue
1,2026-09-01 Tue,2026-12-01 Tue,2026-12-01 Tue,91,0.252778,2026-09-01 Tue,2026-12-01 Tue
2,2026-12-01 Tue,2027-03-01 Mon,2027-03-01 Mon,90,0.250000,2026-12-01 Tue,2027-03-01 Mon


back stub:


,accrual_start,accrual_end,payment,days,frac,reset_start,reset_end
6,2027-12-01 Wed,2028-03-01 Wed,2028-03-01 Wed,91,0.252778,2027-12-01 Wed,2028-03-01 Wed
7,2028-03-01 Wed,2028-06-01 Thu,2028-06-01 Thu,92,0.255556,2028-03-01 Wed,2028-06-01 Thu
8,2028-06-01 Thu,2028-07-17 Mon,2028-07-17 Mon,46,0.127778,2028-06-01 Thu,2028-07-17 Mon


## 2. Single-fixing reset windows

A simple floating coupon reads one forward rate per period. The window it reads depends on the fixing type:

- **Arrears** (the term-rate convention in this library): the window is the accrual period itself.
- **Advance**: the window *ending* at the accrual start, so the first period reads a synthetic window one reset term before the effective date.

A **slower reset** than the payment frequency (pay quarterly, reset annually) maps several consecutive periods to the same window: the rate is set once a year and paid four times.

In [5]:
common = dict(frequency=Frequency.Quarterly, day_count_method=DayCountMethod.Actual360,
              bdc=BDC.ModifiedFollowing, calendar="demo")
start, end = Date(2026, 6, 1), Date(2028, 6, 1)

arrears = build_payment_schedule(start, end, fixing_type=FixingType.Arrears, **common)
advance = build_payment_schedule(start, end, fixing_type=FixingType.Advance, **common)
slow    = build_payment_schedule(start, end, reset_frequency=Frequency.Annually, fixing_type=FixingType.Arrears, **common)

pd.DataFrame({
    "accrual_start": frame(arrears)["accrual_start"],
    "accrual_end": frame(arrears)["accrual_end"],
    "arrears: reset_start": frame(arrears)["reset_start"],
    "arrears: reset_end": frame(arrears)["reset_end"],
    "advance: reset_start": frame(advance)["reset_start"],
    "advance: reset_end": frame(advance)["reset_end"],
    "annual reset: reset_start": frame(slow)["reset_start"],
    "annual reset: reset_end": frame(slow)["reset_end"],
})

,accrual_start,accrual_end,arrears: reset_start,arrears: reset_end,advance: reset_start,advance: reset_end,annual reset: reset_start,annual reset: reset_end
0,2026-06-01 Mon,2026-09-01 Tue,2026-06-01 Mon,2026-09-01 Tue,2026-03-02 Mon,2026-06-01 Mon,2026-06-01 Mon,2027-06-02 Wed
1,2026-09-01 Tue,2026-12-01 Tue,2026-09-01 Tue,2026-12-01 Tue,2026-06-01 Mon,2026-09-01 Tue,2026-06-01 Mon,2027-06-02 Wed
2,2026-12-01 Tue,2027-03-01 Mon,2026-12-01 Tue,2027-03-01 Mon,2026-09-01 Tue,2026-12-01 Tue,2026-06-01 Mon,2027-06-02 Wed
3,2027-03-01 Mon,2027-06-02 Wed,2027-03-01 Mon,2027-06-02 Wed,2026-12-01 Tue,2027-03-01 Mon,2026-06-01 Mon,2027-06-02 Wed
4,2027-06-02 Wed,2027-09-01 Wed,2027-06-02 Wed,2027-09-01 Wed,2027-03-01 Mon,2027-06-02 Wed,2027-06-02 Wed,2028-06-01 Thu
5,2027-09-01 Wed,2027-12-01 Wed,2027-09-01 Wed,2027-12-01 Wed,2027-06-02 Wed,2027-09-01 Wed,2027-06-02 Wed,2028-06-01 Thu
6,2027-12-01 Wed,2028-03-01 Wed,2027-12-01 Wed,2028-03-01 Wed,2027-09-01 Wed,2027-12-01 Wed,2027-06-02 Wed,2028-06-01 Thu
7,2028-03-01 Wed,2028-06-01 Thu,2028-03-01 Wed,2028-06-01 Thu,2027-12-01 Wed,2028-03-01 Wed,2027-06-02 Wed,2028-06-01 Thu


## 3. The observation grid: daily compounded SOFR

A compounded (or averaged) coupon reads every business day in the period. `build_observations=True` with a daily reset builds the Tier-2 grid: for the whole leg, one flat array of fixings with `obs_offsets` marking where each period starts. Each fixing has a **read window** (the overnight period whose rate is used) and a **weight** (the accrual fraction it compounds over).

Without lookback or lockout the read window and the accrual sub-period coincide.

In [6]:
sofr = build_payment_schedule(
    Date(2026, 6, 1), Date(2027, 6, 1), Frequency.Quarterly,
    DayCountMethod.Actual360, BDC.ModifiedFollowing, "demo",
    reset_frequency=Frequency.Daily, fixing_type=FixingType.Arrears, build_observations=True,
)
print(f"{sofr.n_periods} periods, {sofr.obs_weights.size} daily fixings; period offsets into the flat arrays: {sofr.obs_offsets.tolist()}")

def obs_frame(read_starts, read_ends, weights, sub_starts=None, sub_ends=None) -> pd.DataFrame:
    df = pd.DataFrame({"read_start": read_starts.astype(str), "read_end": read_ends.astype(str), "weight": weights.round(6)})
    if sub_starts is not None:
        df.insert(0, "accrues_to", sub_ends.astype(str))
        df.insert(0, "accrues_from", sub_starts.astype(str))
    return df

first = slice(sofr.obs_offsets[0], sofr.obs_offsets[1])
print("first period, first and last fixings (note the 3-day weight over the weekend and the Labor Day holiday):")
display(pd.concat([
    obs_frame(sofr.obs_read_starts[first], sofr.obs_read_ends[first], sofr.obs_weights[first]).head(5),
    obs_frame(sofr.obs_read_starts[first], sofr.obs_read_ends[first], sofr.obs_weights[first]).tail(3),
]))

4 periods, 260 daily fixings; period offsets into the flat arrays: [0, 66, 130, 194]
first period, first and last fixings (note the 3-day weight over the weekend and the Labor Day holiday):


,read_start,read_end,weight
0,2026-06-01,2026-06-02,0.002778
1,2026-06-02,2026-06-03,0.002778
2,2026-06-03,2026-06-04,0.002778
3,2026-06-04,2026-06-05,0.002778
4,2026-06-05,2026-06-08,0.008333
63,2026-08-27,2026-08-28,0.002778
64,2026-08-28,2026-08-31,0.008333
65,2026-08-31,2026-09-01,0.002778


### Lookback, observation shift and lockout

Three conventions for giving the paying agent time to compute the rate. Shown on the first period's accruals with a 5-business-day shift:

- **Lookback** (ISDA): the rate is read 5 business days earlier, but the **weight** is still the real accrual sub-period.
- **Observation shift**: both the rate and the weight come from the shifted window.
- **Lockout**: the last 2 fixings of the period re-use the rate fixed 2 days before the end (the read window stops moving).

The tails of the first period show the difference: compare `accrues_from/to` with `read_start/end`, and look at the last two rows under lockout.

In [7]:
acc_s, acc_e = sofr.accrual_starts[1:2], sofr.accrual_ends[1:2]      # the Sep-Dec period: it contains Labor Day (7 Sep)
lookback = build_observation_grid(acc_s, acc_e, "demo", lookback=5, lookback_style=LookbackStyle.Lookback)
shifted  = build_observation_grid(acc_s, acc_e, "demo", lookback=5, lookback_style=LookbackStyle.ObservationShift)
lockout  = build_observation_grid(acc_s, acc_e, "demo", lockout=2)

for label, g in (("lookback 5B (ISDA)", lookback), ("observation shift 5B", shifted), ("lockout 2B", lockout)):
    print(f"{label}: {g.n_obs} fixings, weights sum to {g.weights.sum():.6f} (period fraction {sofr.period_fracs[1]:.6f})")

# Lookback and observation shift read the same rate; they differ only in the weight, and only where the
# shifted window is a different length from the real sub-period, i.e. around a weekend or holiday.
differ = np.flatnonzero(~np.isclose(lookback.weights, shifted.weights))
print()
print(f"fixings where the two styles weight differently: {differ.tolist()}")
display(pd.DataFrame({
    "accrues_from": lookback.sub_starts[differ].astype(str), "accrues_to": lookback.sub_ends[differ].astype(str),
    "read_start": lookback.read_starts[differ].astype(str), "read_end": lookback.read_ends[differ].astype(str),
    "lookback weight": lookback.weights[differ].round(6), "obs-shift weight": shifted.weights[differ].round(6),
}))
print("lockout: the last two fixings re-use the rate fixed two business days before the period end")
display(obs_frame(lockout.read_starts, lockout.read_ends, lockout.weights, lockout.sub_starts, lockout.sub_ends).tail(4))

lookback 5B (ISDA): 64 fixings, weights sum to 0.252778 (period fraction 0.252778)
observation shift 5B: 64 fixings, weights sum to 0.252778 (period fraction 0.252778)
lockout 2B: 64 fixings, weights sum to 0.252778 (period fraction 0.252778)

fixings where the two styles weight differently: [3, 7, 8]


,accrues_from,accrues_to,read_start,read_end,lookback weight,obs-shift weight
0,2026-09-04,2026-09-08,2026-08-28,2026-08-31,0.011111,0.008333
1,2026-09-11,2026-09-14,2026-09-03,2026-09-04,0.008333,0.002778
2,2026-09-14,2026-09-15,2026-09-04,2026-09-08,0.002778,0.011111


lockout: the last two fixings re-use the rate fixed two business days before the period end


,accrues_from,accrues_to,read_start,read_end,weight
60,2026-11-25,2026-11-26,2026-11-25,2026-11-26,0.002778
61,2026-11-26,2026-11-27,2026-11-26,2026-11-27,0.002778
62,2026-11-27,2026-11-30,2026-11-27,2026-11-30,0.008333
63,2026-11-30,2026-12-01,2026-11-27,2026-11-30,0.002778


## 4. Monthly fixings inside a quarterly coupon

A term-rate compounder: the coupon pays quarterly but compounds three monthly fixings. The sub-periods come from a monthly schedule rather than business days.

This is where `lookback_style` matters even with no lookback. `Lookback` (the ISDA daily-RFR convention) reads the **overnight** rate at each sub-period start, so with monthly sub-periods it would read a one-day window and weight it by a month, which is wrong for a term fixing. `ObservationShift` reads the **whole** sub-period, which is what a 1M term rate is. The builder defaults to `Lookback`, so a term compounder must say so.

In [8]:
kwargs = dict(frequency=Frequency.Quarterly, day_count_method=DayCountMethod.Actual360, bdc=BDC.ModifiedFollowing,
              calendar="demo", reset_frequency=Frequency.Monthly, fixing_type=FixingType.Arrears, build_observations=True)
daily_style = build_payment_schedule(Date(2026, 6, 1), Date(2027, 6, 1), **kwargs)                                      # default: Lookback
term = build_payment_schedule(Date(2026, 6, 1), Date(2027, 6, 1), lookback_style=LookbackStyle.ObservationShift, **kwargs)
print(f"{term.n_periods} quarterly periods, {term.obs_weights.size} monthly fixings, offsets {term.obs_offsets.tolist()}")
pd.DataFrame({
    "accrues_from": term.obs_read_starts.astype(str),
    "accrues_to": term.obs_read_ends.astype(str),
    "weight": term.obs_weights.round(6),
    "Lookback style would read": daily_style.obs_read_starts.astype(str) + " -> " + daily_style.obs_read_ends.astype(str),
})

4 quarterly periods, 12 monthly fixings, offsets [0, 3, 6, 9]


,accrues_from,accrues_to,weight,Lookback style would read
0,2026-06-01,2026-07-01,0.083333,2026-06-01 -> 2026-06-02
1,2026-07-01,2026-08-03,0.091667,2026-07-01 -> 2026-07-02
2,2026-08-03,2026-09-01,0.080556,2026-08-03 -> 2026-08-04
3,2026-09-01,2026-10-01,0.083333,2026-09-01 -> 2026-09-02
4,2026-10-01,2026-11-02,0.088889,2026-10-01 -> 2026-10-02
5,2026-11-02,2026-12-01,0.080556,2026-11-02 -> 2026-11-03
6,2026-12-01,2027-01-01,0.086111,2026-12-01 -> 2026-12-02
7,2027-01-01,2027-02-01,0.086111,2027-01-01 -> 2027-01-04
8,2027-02-01,2027-03-01,0.077778,2027-02-01 -> 2027-02-02
9,2027-03-01,2027-04-01,0.086111,2027-03-01 -> 2027-03-02


## 5. What the kernel does with it

The grid is flat on purpose. The compiler concatenates every leg's fixings into one array and keeps the offsets, so a compounded coupon is one vectorised pass over all fixings and one `reduceat` per period, with no Python loop over periods or legs. The weights of a period's fixings sum to its accrual fraction, which is the check the engines rely on.

In [9]:
ends = np.append(sofr.obs_offsets, sofr.obs_weights.size)
per_period = np.add.reduceat(sofr.obs_weights, sofr.obs_offsets)
pd.DataFrame({
    "period": range(sofr.n_periods),
    "fixings": np.diff(ends),
    "sum of weights": per_period.round(8),
    "period_frac": sofr.period_fracs.round(8),
    "match": np.isclose(per_period, sofr.period_fracs),
})

,period,fixings,sum of weights,period_frac,match
0,0,66,0.255556,0.255556,True
1,1,64,0.252778,0.252778,True
2,2,64,0.250000,0.250000,True
3,3,66,0.258333,0.258333,True
